# GRNTI dataset EDA

Exploratory analysis of the pre-processed GRNTI text-classification splits.
Run cells top-to-bottom after completing the data pipeline (Task 3/4).

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
proc = REPO / "data" / "processed"

train = pd.read_parquet(proc / "train.parquet")
val   = pd.read_parquet(proc / "val.parquet")
test  = pd.read_parquet(proc / "test.parquet")
encoder = json.loads((proc / "label_encoder.json").read_text(encoding="utf-8"))
idx_to_text = {int(k): v for k, v in encoder["idx_to_text"].items()}

print(f"train={len(train)} val={len(val)} test={len(test)} classes={encoder['num_classes']}")

## Class distribution

In [ ]:
counts = (
    train["label_idx"]
    .map(idx_to_text)
    .value_counts()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(counts.index[::-1], counts.values[::-1])
ax.set_xlabel("Count")
ax.set_title("Class distribution (train split)")
plt.tight_layout()
plt.show()

## Text length distribution (characters)

In [ ]:
lengths = train["text"].str.len()
median_len = lengths.median()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lengths, bins=50, edgecolor="white")
ax.axvline(median_len, color="red", linestyle="--", label=f"median={median_len:.0f}")
ax.set_xlabel("Characters")
ax.set_ylabel("Count")
ax.set_title("Text length in characters (train)")
ax.legend()
plt.tight_layout()
plt.show()

## Sample abstract per class (top 5 classes)

In [ ]:
top5_classes = (
    train["label_idx"]
    .value_counts()
    .head(5)
    .index
    .tolist()
)

for cls_idx in top5_classes:
    cls_name = idx_to_text[cls_idx]
    sample = (
        train.loc[train["label_idx"] == cls_idx, "text"]
        .sample(1, random_state=42)
        .iloc[0]
    )
    print(f"--- {cls_name} ---")
    print(sample[:300])
    print()